# SILVER LAYER (4 TABLES)

## S1) products — SCD Type 1 from CDC feed

**Idempotent.**
- Rank every change by `seqNum` per `product_id`, keep rank 1 (the latest known state).
- If that latest change is a DELETE → delete the row from silver.
- Otherwise upsert, and only when the incoming `seq_num` is newer than the stored one.
- Replaying the same files changes nothing, because the seq_num guard blocks the update.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "abs_dev"
db_name = "abs_db"

# ============================================================
# 1. CREATE SCHEMA
# ============================================================

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {catalog}.{db_name}
""")


# ============================================================
# 2. CREATE TARGET PRODUCTS TABLE
# ============================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.products (
    product_id       STRING,
    product_name     STRING,
    product_category STRING,
    product_price    DECIMAL(12,2),
    seq_num          BIGINT,
    updated_at       TIMESTAMP
)
USING DELTA
""")


# ============================================================
# 3. READ AND CLEAN CDC DATA FROM BRONZE
# ============================================================

prod_cdc = (
    spark.read.table(f"{catalog}.{db_name}.products_bz")
        .filter(F.col("product_id").isNotNull())
        .select(
            F.col("product_id"),
            F.col("product_name"),
            F.col("product_category"),

            F.col("product_price")
                .cast("decimal(12,2)")
                .alias("product_price"),

            F.upper(F.trim(F.col("operation")))
                .alias("operation"),

            F.col("seqNum")
                .cast("bigint")
                .alias("seq_num")
        )
        .filter(F.col("seq_num").isNotNull())
        .dropDuplicates(["product_id", "seq_num"])
)


# ============================================================
# 4. KEEP ONLY THE LATEST CDC EVENT PER PRODUCT
# ============================================================

window = (
    Window
        .partitionBy("product_id")
        .orderBy(F.col("seq_num").desc())
)

prod_latest = (
    prod_cdc
        .withColumn("row_num", F.row_number().over(window))
        .filter(F.col("row_num") == 1)
        .drop("row_num")
        .withColumn("updated_at", F.current_timestamp())
)


# ============================================================
# 5. CREATE TEMPORARY SOURCE VIEW
# ============================================================

prod_latest.createOrReplaceTempView("products_cdc_source")


# ============================================================
# 6. MERGE CDC INTO TARGET
# ============================================================

spark.sql(f"""
MERGE INTO {catalog}.{db_name}.products AS target
USING products_cdc_source AS source
ON target.product_id = source.product_id

-- DELETE only when the incoming CDC event is newer
WHEN MATCHED
     AND source.operation IN ('DELETE', 'D')
     AND target.seq_num < source.seq_num
THEN DELETE

-- UPDATE only when the incoming event is newer
WHEN MATCHED
     AND source.operation NOT IN ('DELETE', 'D')
     AND target.seq_num < source.seq_num
THEN UPDATE SET
    target.product_name     = source.product_name,
    target.product_category = source.product_category,
    target.product_price    = source.product_price,
    target.seq_num          = source.seq_num,
    target.updated_at       = source.updated_at

-- INSERT new products, but never insert DELETE events
WHEN NOT MATCHED
     AND source.operation NOT IN ('DELETE', 'D')
THEN INSERT (
    product_id,
    product_name,
    product_category,
    product_price,
    seq_num,
    updated_at
)
VALUES (
    source.product_id,
    source.product_name,
    source.product_category,
    source.product_price,
    source.seq_num,
    source.updated_at
)
""")


# ============================================================
# 7. DISPLAY RESULT
# ============================================================

display(
    spark.table(f"{catalog}.{db_name}.products")
         .orderBy(F.col("updated_at").desc())
         .limit(20)
)

product_id,product_name,product_category,product_price,seq_num,updated_at
2,rise,Electronics,236.01,1,2026-09-28T10:46:10.991939Z
3,where,Home Goods,555.38,3,2026-09-28T10:46:10.991939Z
4,economic,Electronics,123.91,2,2026-09-28T10:46:10.991939Z
5,decide,Home Goods,201.97,2,2026-09-28T10:46:10.991939Z


## S2) customers — SCD Type 1 from CDC feed

**Idempotent.** Identical logic to `products`, driven by `sequenceNum`.
Rows with a null `customer_id` or an unparseable sequence number are dropped rather than
merged — a null key would collapse unrelated customers into one row.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "abs_dev"
db_name = "abs_db"


# ============================================================
# 1. CREATE CUSTOMERS TARGET TABLE
# ============================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.customers (
    customer_id     STRING,
    customer_name   STRING,
    customer_email  STRING,
    customer_city   STRING,
    customer_state  STRING,
    seq_num         BIGINT,
    updated_at      TIMESTAMP
)
USING DELTA
""")


# ============================================================
# 2. READ AND CLEAN CDC DATA FROM BRONZE
# ============================================================

cust_cdc = (
    spark.read.table(f"{catalog}.{db_name}.customers_bz")
        .filter(F.col("customer_id").isNotNull())
        .select(
            F.col("customer_id"),
            F.col("customer_name"),

            F.lower(
                F.trim(F.col("customer_email"))
            ).alias("customer_email"),

            F.col("customer_city"),

            F.upper(
                F.trim(F.col("customer_state"))
            ).alias("customer_state"),

            F.upper(
                F.trim(F.col("operation"))
            ).alias("operation"),

            F.col("sequenceNum")
                .cast("bigint")
                .alias("seq_num")
        )
        .filter(F.col("seq_num").isNotNull())
        .dropDuplicates(["customer_id", "seq_num"])
)


# ============================================================
# 3. KEEP ONLY THE LATEST CDC EVENT PER CUSTOMER
# ============================================================

window = (
    Window
        .partitionBy("customer_id")
        .orderBy(F.col("seq_num").desc())
)

cust_latest = (
    cust_cdc
        .withColumn("row_num", F.row_number().over(window))
        .filter(F.col("row_num") == 1)
        .drop("row_num")
        .withColumn("updated_at", F.current_timestamp())
)


# ============================================================
# 4. CREATE TEMPORARY CDC SOURCE VIEW
# ============================================================

cust_latest.createOrReplaceTempView("customers_cdc_source")


# ============================================================
# 5. MERGE CDC INTO CUSTOMERS
# ============================================================

spark.sql(f"""
MERGE INTO {catalog}.{db_name}.customers AS target
USING customers_cdc_source AS source
ON target.customer_id = source.customer_id

-- DELETE existing customer only if the incoming event is newer
WHEN MATCHED
     AND source.operation IN ('DELETE', 'D')
     AND target.seq_num < source.seq_num
THEN DELETE

-- UPDATE existing customer only if the incoming event is newer
WHEN MATCHED
     AND source.operation NOT IN ('DELETE', 'D')
     AND target.seq_num < source.seq_num
THEN UPDATE SET
    target.customer_name  = source.customer_name,
    target.customer_email = source.customer_email,
    target.customer_city  = source.customer_city,
    target.customer_state = source.customer_state,
    target.seq_num        = source.seq_num,
    target.updated_at     = source.updated_at

-- INSERT new customer records
WHEN NOT MATCHED
     AND source.operation NOT IN ('DELETE', 'D')
THEN INSERT (
    customer_id,
    customer_name,
    customer_email,
    customer_city,
    customer_state,
    seq_num,
    updated_at
)
VALUES (
    source.customer_id,
    source.customer_name,
    source.customer_email,
    source.customer_city,
    source.customer_state,
    source.seq_num,
    source.updated_at
)
""")


# ============================================================
# 6. DISPLAY RESULT
# ============================================================

display(
    spark.table(f"{catalog}.{db_name}.customers")
         .orderBy(F.col("updated_at").desc())
         .limit(20)
)

customer_id,customer_name,customer_email,customer_city,customer_state,seq_num,updated_at
1,Paul Shelton,heathermartinez@example.net,Smithborough,AK,1,2026-09-28T10:50:11.370856Z
2,Brandon Brooks,debraandersen@example.com,New York,NY,2,2026-09-28T10:50:11.370856Z
4,Derek Hawkins,porterjasmine@example.com,Martinezshire,ND,1,2026-09-28T10:50:11.370856Z
5,Beth Turner,ujohnson@example.net,Priceton,CO,1,2026-09-28T10:50:11.370856Z
6,Robert,robert@example.com,California,SF,2,2026-09-28T10:50:11.370856Z


## S3) sales — typed fact from sales_bz

**Idempotent.** The grain is one line item: `order_id` + `product_id` + `transaction_id`.
A sale that already landed never changes, so the MERGE is insert-only and re-running the
notebook over the same bronze data inserts nothing.

**Amount convention:** `total_amount` is treated as the *gross* line value and
`net_amount = total_amount - discount_amount`. If your source already nets the discount out,
delete the `net_amount` expression and alias `total_amount` instead.

In [0]:
from pyspark.sql import functions as F


# ============================================================
# 1. CREATE SALES TARGET TABLE
# ============================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.sales (
    order_id         STRING,
    customer_id      STRING,
    transaction_id   STRING,
    product_id       STRING,
    quantity         INT,
    discount_amount  DECIMAL(12,2),
    total_amount     DECIMAL(12,2),
    net_amount       DECIMAL(12,2),
    order_ts         TIMESTAMP,
    order_date       DATE
)
USING DELTA
""")


# ============================================================
# 2. READ AND CLEAN SALES DATA FROM BRONZE
# ============================================================

sales_src = (
    spark.read.table(f"{catalog}.{db_name}.sales_bz")
        .select(
            F.col("order_id").cast("string").alias("order_id"),
            F.col("customer_id").cast("string").alias("customer_id"),
            F.col("transaction_id").cast("string").alias("transaction_id"),
            F.col("product_id").cast("string").alias("product_id"),

            F.col("quantity")
                .cast("int")
                .alias("quantity"),

            F.coalesce(
                F.col("discount_amount").cast("decimal(12,2)"),
                F.lit(0).cast("decimal(12,2)")
            ).alias("discount_amount"),

            F.col("total_amount")
                .cast("decimal(12,2)")
                .alias("total_amount"),

            F.to_timestamp(
                F.to_date(F.col("order_date"), "M/d/yyyy")
            ).alias("order_ts")
        )

        # --------------------------------------------
        # Basic data-quality filters
        # --------------------------------------------
        .filter(F.col("order_id").isNotNull())
        .filter(F.col("product_id").isNotNull())
        .filter(F.col("transaction_id").isNotNull())
        .filter(F.col("quantity") > 0)
        .filter(F.col("total_amount").isNotNull())
        .filter(F.col("order_ts").isNotNull())

        # --------------------------------------------
        # Derive net amount and order date
        # --------------------------------------------
        .withColumn(
            "net_amount",
            (
                F.col("total_amount") -
                F.col("discount_amount")
            ).cast("decimal(12,2)")
        )

        .withColumn(
            "order_date",
            F.to_date(F.col("order_ts"))
        )

        .select(
            "order_id",
            "customer_id",
            "transaction_id",
            "product_id",
            "quantity",
            "discount_amount",
            "total_amount",
            "net_amount",
            "order_ts",
            "order_date"
        )

        # --------------------------------------------
        # Prevent duplicate business keys
        # --------------------------------------------
        .dropDuplicates(
            ["order_id", "product_id", "transaction_id"]
        )
)


# ============================================================
# 3. CREATE TEMPORARY SOURCE VIEW
# ============================================================

sales_src.createOrReplaceTempView("sales_source")


# ============================================================
# 4. INCREMENTAL MERGE INTO SALES
# ============================================================

spark.sql(f"""
MERGE INTO {catalog}.{db_name}.sales AS target
USING sales_source AS source

ON  target.order_id = source.order_id
AND target.product_id = source.product_id
AND target.transaction_id = source.transaction_id

WHEN NOT MATCHED THEN
INSERT (
    order_id,
    customer_id,
    transaction_id,
    product_id,
    quantity,
    discount_amount,
    total_amount,
    net_amount,
    order_ts,
    order_date
)
VALUES (
    source.order_id,
    source.customer_id,
    source.transaction_id,
    source.product_id,
    source.quantity,
    source.discount_amount,
    source.total_amount,
    source.net_amount,
    source.order_ts,
    source.order_date
)
""")


# ============================================================
# 5. DISPLAY RESULT
# ============================================================

display(
    spark.table(f"{catalog}.{db_name}.sales")
         .orderBy(F.col("order_ts").desc())
         .limit(20)
)

order_id,customer_id,transaction_id,product_id,quantity,discount_amount,total_amount,net_amount,order_ts,order_date
3,1,1003,5,2,14.78,389.16,374.38,2023-12-05T00:00:00Z,2023-12-05
50,4,1050,5,4,1.21,806.67,805.46,2023-12-05T00:00:00Z,2023-12-05
15,3,1015,2,5,15.61,1164.44,1148.83,2023-12-05T00:00:00Z,2023-12-05
40,5,1040,4,2,1.71,246.11,244.40,2023-12-05T00:00:00Z,2023-12-05
17,4,1017,4,4,6.55,489.09,482.54,2023-12-05T00:00:00Z,2023-12-05
27,5,1027,3,4,18.17,1727.35,1709.18,2023-12-05T00:00:00Z,2023-12-05
32,2,1032,3,5,6.58,2175.32,2168.74,2023-12-05T00:00:00Z,2023-12-05
4,5,1004,5,3,11.93,593.98,582.05,2023-12-04T00:00:00Z,2023-12-04
33,1,1033,4,5,18.07,601.48,583.41,2023-12-04T00:00:00Z,2023-12-04
37,5,1037,2,3,14.25,693.78,679.53,2023-12-04T00:00:00Z,2023-12-04


## S4) sales_enriched — fact joined to both dimensions

**Idempotent.** Insert-only on the line grain. Dimensions are joined with `left` so a sale
for a product or customer that has not arrived yet is still kept (with null attributes)
rather than silently dropped from revenue.

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.sales_enriched (
  order_id         STRING,
  transaction_id   STRING,
  order_date       DATE,
  customer_id      STRING,
  customer_name    STRING,
  customer_city    STRING,
  customer_state   STRING,
  product_id       STRING,
  product_name     STRING,
  product_category STRING,
  product_price    DECIMAL(12,2),
  quantity         INT,
  discount_amount  DECIMAL(12,2),
  total_amount     DECIMAL(12,2),
  net_amount       DECIMAL(12,2)
) USING DELTA
""")

df_sales     = spark.read.table(f"{catalog}.{db_name}.sales")
df_products  = spark.read.table(f"{catalog}.{db_name}.products").drop("seq_num", "updated_at")
df_customers = spark.read.table(f"{catalog}.{db_name}.customers").drop("seq_num", "updated_at")

sales_enriched = (
    df_sales
        .join(df_customers, "customer_id", "left")
        .join(df_products,  "product_id",  "left")
        .select("order_id", "transaction_id", "order_date",
                "customer_id", "customer_name", "customer_city", "customer_state",
                "product_id", "product_name", "product_category", "product_price",
                "quantity", "discount_amount", "total_amount", "net_amount")
)

sales_enriched.createOrReplaceTempView("sales_enriched_source")

spark.sql(f"""
MERGE INTO {catalog}.{db_name}.sales_enriched a
USING sales_enriched_source b
ON a.order_id = b.order_id AND a.product_id = b.product_id AND a.transaction_id = b.transaction_id
WHEN NOT MATCHED THEN INSERT *
""")

display(spark.table(f"{catalog}.{db_name}.sales_enriched").limit(20))

order_id,transaction_id,order_date,customer_id,customer_name,customer_city,customer_state,product_id,product_name,product_category,product_price,quantity,discount_amount,total_amount,net_amount
10,1010,2023-12-03,1,Paul Shelton,Smithborough,AK,3,where,Home Goods,555.38,2,11.59,861.17,849.58
33,1033,2023-12-04,1,Paul Shelton,Smithborough,AK,4,economic,Electronics,123.91,5,18.07,601.48,583.41
34,1034,2023-12-04,5,Beth Turner,Priceton,CO,4,economic,Electronics,123.91,4,5.69,489.95,484.26
39,1039,2023-12-02,1,Paul Shelton,Smithborough,AK,5,decide,Home Goods,201.97,5,1.64,1008.21,1006.57
1,1001,2023-12-02,1,Paul Shelton,Smithborough,AK,3,where,Home Goods,555.38,3,9.02,1300.12,1291.10
3,1003,2023-12-05,1,Paul Shelton,Smithborough,AK,5,decide,Home Goods,201.97,2,14.78,389.16,374.38
12,1012,2023-12-01,2,Brandon Brooks,New York,NY,5,decide,Home Goods,201.97,4,1.10,806.78,805.68
16,1016,2023-12-03,1,Paul Shelton,Smithborough,AK,5,decide,Home Goods,201.97,2,6.81,397.13,390.32
25,1025,2023-12-01,2,Brandon Brooks,New York,NY,4,economic,Electronics,123.91,3,13.84,357.89,344.05
50,1050,2023-12-05,4,Derek Hawkins,Martinezshire,ND,5,decide,Home Goods,201.97,4,1.21,806.67,805.46
